# HydroWatch Amur — финальное решение

Ноутбук строит воспроизводимый `submission.csv` и GeoTIFF-маски без Google Earth Engine и Google Cloud. Sentinel-1 загружается из публичного STAC Microsoft Planetary Computer, Sentinel-2 — из Earth Search Element 84. Основная модель — компактная physics-aware нейросеть по пиксельным и многомасштабным признакам.

Главная защита от переобучения — **leave-one-event-out**: для каждой открытой пары прогноз делает модель, которая не видела reference-маски этого события. Если в `sample_submission.csv` появятся новые пары без reference, для них автоматически обучается финальная модель на всех доступных событиях.


In [1]:
# 1. Зависимости

import importlib.util
import subprocess
import sys

missing = []
if importlib.util.find_spec("pystac_client") is None:
    missing.append("pystac-client>=0.8,<0.10")
if importlib.util.find_spec("planetary_computer") is None:
    missing.append("planetary-computer>=1.0,<2")

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Зависимости готовы")


Зависимости готовы


**Зачем:** ставим только один дополнительный пакет — STAC-клиент. Мы специально не переустанавливаем NumPy, SciPy и PyTorch, поэтому не ломаем стандартную среду Colab и не требуем перезапуска runtime.


In [2]:
# 2. Импорты

import gc
import json
import math
import os
import random
import shutil
import time
import warnings
import zipfile
from dataclasses import asdict, dataclass
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import planetary_computer
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from pyproj import Transformer
from pystac_client import Client
from rasterio.features import rasterize
from rasterio.transform import Affine
from rasterio.warp import Resampling, reproject
from shapely.geometry import shape
from shapely.ops import transform as shp_transform
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


**Зачем:** здесь только импорты, без скрытой логики. В решении нет SciPy, scikit-image, geemap и Earth Engine — это уменьшает число конфликтов и делает запуск стабильнее.


In [3]:
# 3. Настройки

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")


@dataclass
class CFG:
    work_dir: str = "/content/hydrowatch_final"
    work_res_m: int = 20
    out_res_m: int = 10
    pixels_per_scene: int = 28000
    epochs: int = 12
    batch_size: int = 8192
    lr: float = 1.5e-3
    weight_decay: float = 2e-4
    predict_batch: int = 200000
    feature_version: int = 7
    seed: int = 42


cfg = CFG()

WORK = Path(cfg.work_dir)
DATA = WORK / "data"
CACHE = WORK / f"features_v{cfg.feature_version}"
OOF = WORK / "oof"
MODELS = WORK / "models"
OUT = WORK / "result"
PRED = OUT / "predictions"

for path in [WORK, DATA, CACHE, OOF, MODELS, OUT, PRED]:
    path.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Устройство:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM, ГБ:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("Рабочая папка:", WORK)


Устройство: cuda
GPU: Tesla T4
VRAM, ГБ: 14.56
Рабочая папка: /content/hydrowatch_final


**Зачем:** фиксируем seed и все параметры в одном месте. Разрешение 20 м сильно ускоряет обучение и инференс, а финальные маски затем возвращаются на обязательную сетку 10 м.


In [4]:
# 4. Датасет

try:
    from google.colab import files
except Exception:
    files = None

zip_path = WORK / "hydrowatch_amur_dataset_lite.zip"

if not zip_path.exists():
    candidates = [
        Path("/content/hydrowatch_amur_dataset_lite.zip"),
        *Path("/content").glob("*hydrowatch*.zip"),
    ]
    candidates = [p for p in candidates if p.exists() and p.is_file()]

    if candidates:
        shutil.copy2(candidates[0], zip_path)
        print("Использую архив:", candidates[0].name)
    elif files is not None:
        print("Выбери hydrowatch_amur_dataset_lite.zip")
        uploaded = files.upload()
        names = [name for name in uploaded if name.lower().endswith(".zip")]
        if not names:
            raise FileNotFoundError("ZIP-файл не загружен")
        zip_path.write_bytes(uploaded[names[0]])
    else:
        raise FileNotFoundError("Положи hydrowatch_amur_dataset_lite.zip рядом с ноутбуком")

if not zipfile.is_zipfile(zip_path):
    raise RuntimeError("Загруженный файл не является корректным ZIP")

ROOT = DATA / "hydrowatch_amur"

if not (ROOT / "pairs.csv").exists():
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(DATA)

pairs = pd.read_csv(ROOT / "pairs.csv")
sample = pd.read_csv(ROOT / "sample_submission.csv")
scene_catalog = pd.read_csv(ROOT / "tables" / "scene_catalog.csv")

aoi_geo = json.loads((ROOT / "vectors" / "aoi.geojson").read_text(encoding="utf-8"))
aoi_features = {f["properties"]["aoi_id"]: f for f in aoi_geo["features"]}

hydro_geo = json.loads((ROOT / "vectors" / "hydrography_osm.geojson").read_text(encoding="utf-8"))
hydro_by_aoi = {}
for feature in hydro_geo["features"]:
    hydro_by_aoi.setdefault(feature["properties"].get("aoi_id"), []).append(feature)

if "reference_mask" not in pairs.columns:
    pairs["reference_mask"] = ""

pairs["has_reference"] = pairs["reference_mask"].apply(
    lambda x: isinstance(x, str) and bool(x.strip()) and (ROOT / x).exists()
)

row_by_pid = pairs.set_index("pair_id")
unknown = set(sample["pair_id"]) - set(pairs["pair_id"])
if unknown:
    raise ValueError(f"В sample_submission есть неизвестные pair_id: {sorted(unknown)}")

print("Пар:", len(pairs))
print("С reference:", int(pairs["has_reference"].sum()))
print("Без reference:", int((~pairs["has_reference"]).sum()))
display(pairs[["pair_id", "event_id", "event_kind", "date_pre_sar", "date_peak_sar", "date_pre_opt", "date_peak_opt", "has_reference"]])


Использую архив: hydrowatch_amur_dataset_lite.zip
Пар: 11
С reference: 11
Без reference: 0


,pair_id,event_id,event_kind,date_pre_sar,date_peak_sar,date_pre_opt,date_peak_opt,has_reference
0,baseline_2018_09_low__blagoveshchensk,baseline_2018_09_low,baseline,2018-07-29,2018-09-15,NaN,NaN,True
1,baseline_2018_09_low__konstantinovka,baseline_2018_09_low,baseline,2018-07-29,2018-09-15,NaN,NaN,True
2,baseline_2018_09_low__svobodny,baseline_2018_09_low,baseline,2018-07-29,2018-09-15,NaN,NaN,True
3,flood_2019_07_amur__belogorsk,flood_2019_07_amur,rain_flood,2019-06-13,2019-07-25,2019-06-18,2019-07-30,True
4,flood_2019_07_amur__blagoveshchensk,flood_2019_07_amur,rain_flood,2019-06-13,2019-07-25,NaN,NaN,True
5,flood_2019_07_amur__konstantinovka,flood_2019_07_amur,rain_flood,2019-06-13,2019-07-25,NaN,NaN,True
6,flood_2019_07_amur__svobodny,flood_2019_07_amur,rain_flood,2019-06-13,2019-07-25,2019-06-18,2019-07-30,True
7,flood_2021_06_amur__blagoveshchensk,flood_2021_06_amur,rain_flood,2021-05-14,2021-07-01,2021-05-18,2021-06-27,True
8,flood_2021_06_amur__konstantinovka,flood_2021_06_amur,rain_flood,2021-05-14,2021-07-01,2021-05-18,2021-06-27,True
9,flood_2021_06_amur__poyarkovo,flood_2021_06_amur,rain_flood,2021-05-14,2021-07-01,2021-05-18,2021-06-27,True


**Зачем:** ноутбук сам находит загруженный ZIP и больше не зависит от `gdown`. Reference используется только как обучающая разметка и для общей OOF-калибровки, а не для копирования ответа.


In [5]:
# 5. Сетка и маска AOI

TO_UTM = Transformer.from_crs("EPSG:4326", "EPSG:32652", always_xy=True).transform


@lru_cache(maxsize=None)
def aoi_utm(aoi_id):
    return shp_transform(TO_UTM, shape(aoi_features[aoi_id]["geometry"]))


def grid_for_aoi(aoi_id, res):
    geom = aoi_utm(aoi_id)
    minx, miny, maxx, maxy = geom.bounds
    left = math.floor(minx / res) * res
    bottom = math.floor(miny / res) * res
    right = math.ceil(maxx / res) * res
    top = math.ceil(maxy / res) * res
    width = int(round((right - left) / res))
    height = int(round((top - bottom) / res))
    transform = Affine(res, 0, left, 0, -res, top)
    return {
        "height": height,
        "width": width,
        "transform": transform,
        "crs": "EPSG:32652",
        "res": res,
    }


def work_grid(row):
    return grid_for_aoi(row.aoi_id, cfg.work_res_m)


def out_grid(row):
    return grid_for_aoi(row.aoi_id, cfg.out_res_m)


@lru_cache(maxsize=None)
def aoi_mask(aoi_id, res):
    grid = grid_for_aoi(aoi_id, res)
    return rasterize(
        [(aoi_utm(aoi_id), 1)],
        out_shape=(grid["height"], grid["width"]),
        transform=grid["transform"],
        fill=0,
        dtype="uint8",
    ).astype(bool)


checks = []
for _, row in pairs[pairs["has_reference"]].iterrows():
    grid = out_grid(row)
    with rasterio.open(ROOT / row.reference_mask) as src:
        ok = (
            src.height == grid["height"]
            and src.width == grid["width"]
            and src.transform.almost_equals(grid["transform"], precision=1e-6)
            and src.crs == rasterio.crs.CRS.from_string(grid["crs"])
        )
    checks.append(ok)

if checks:
    print("Сетка:", sum(checks), "/", len(checks), "совпадений")
    if not all(checks):
        raise RuntimeError("Сетка AOI не совпала с метаданными reference")


Сетка: 11 / 11 совпадений


**Зачем:** организаторам нужны GeoTIFF ровно на их 10-метровой сетке. Мы восстанавливаем её из AOI и проверяем только геометрию открытых reference-файлов — значения эталонных пикселей здесь не читаются.


In [6]:
# 6. STAC-каталоги

S1_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
S2_URL = "https://earth-search.aws.element84.com/v1"


def open_catalog(url, modifier=None):
    try:
        client = Client.open(url, modifier=modifier)
        list(client.get_collections())[:1]
        return client
    except Exception as exc:
        print("Каталог временно недоступен:", url)
        print(type(exc).__name__, str(exc)[:180])
        return None


S1_STAC = open_catalog(S1_URL, planetary_computer.sign_inplace)
S2_STAC = open_catalog(S2_URL)

print("Sentinel-1 STAC:", "OK" if S1_STAC is not None else "fallback")
print("Sentinel-2 STAC:", "OK" if S2_STAC is not None else "fallback")


Sentinel-1 STAC: OK
Sentinel-2 STAC: OK


**Зачем:** Sentinel берём без Google Cloud и без Earth Engine. Ошибка внешнего STAC не останавливает ноутбук: в таком случае соответствующая модальность заполняется нейтральными значениями, а модель продолжает работать на AUX и доступных данных.


In [7]:
# 7. Чтение Sentinel в нашу сетку

S2_ALIASES = {
    "green": ["green", "B03", "b03"],
    "red": ["red", "B04", "b04"],
    "nir": ["nir", "B08", "b08", "nir08"],
    "swir": ["swir16", "B11", "b11"],
    "scl": ["scl", "SCL", "scene_classification"],
}


def item_date(item):
    value = item.datetime or item.properties.get("datetime")
    return pd.Timestamp(value).tz_localize(None).normalize()


def asset_href(item, aliases):
    for key in aliases:
        if key in item.assets:
            href = item.assets[key].href
            if href.startswith("https://") or href.startswith("http://"):
                return href
    return None


def read_remote(href, grid, resampling=Resampling.bilinear):
    if not href:
        return None
    dst = np.full((grid["height"], grid["width"]), np.nan, dtype=np.float32)
    try:
        with rasterio.Env(
            GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
            CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.tiff",
            GDAL_HTTP_MULTIRANGE="YES",
            VSI_CACHE="TRUE",
        ):
            with rasterio.open(href) as src:
                reproject(
                    source=rasterio.band(src, 1),
                    destination=dst,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src.nodata,
                    dst_transform=grid["transform"],
                    dst_crs=grid["crs"],
                    dst_nodata=np.nan,
                    resampling=resampling,
                    num_threads=2,
                )
                scale = src.scales[0] if src.scales else 1.0
                offset = src.offsets[0] if src.offsets else 0.0
                if np.isfinite(scale) and (abs(scale - 1.0) > 1e-12 or abs(offset) > 1e-12):
                    dst = dst * float(scale) + float(offset)
        return dst
    except Exception:
        return None


def search_items(client, collection, geometry, date_str, days=1, max_items=80):
    if client is None or pd.isna(date_str):
        return []
    date = pd.Timestamp(date_str).normalize()
    start = (date - pd.Timedelta(days=days)).strftime("%Y-%m-%d")
    end = (date + pd.Timedelta(days=days + 1)).strftime("%Y-%m-%d")
    try:
        search = client.search(
            collections=[collection],
            intersects=geometry,
            datetime=f"{start}/{end}",
            max_items=max_items,
        )
        return list(search.items())
    except Exception:
        return []


def choose_s1_items(row, date_str):
    items = search_items(S1_STAC, "sentinel-1-grd", aoi_features[row.aoi_id]["geometry"], date_str, days=1)
    target = pd.Timestamp(date_str).normalize()
    selected = []

    for item in items:
        props = item.properties
        mode = str(props.get("sar:instrument_mode", "IW")).upper()
        orbit = str(props.get("sat:orbit_state", "")).upper()
        rel = props.get("sat:relative_orbit")
        pols = [str(x).upper() for x in props.get("sar:polarizations", [])]

        if mode and mode != "IW":
            continue
        if row.orbit_pass == row.orbit_pass and orbit and orbit != str(row.orbit_pass).upper():
            continue
        if row.relative_orbit == row.relative_orbit and rel is not None and int(rel) != int(row.relative_orbit):
            continue
        if pols and not {"VV", "VH"}.issubset(set(pols)):
            continue
        if asset_href(item, ["vv", "VV"]) and asset_href(item, ["vh", "VH"]):
            selected.append(item)

    if not selected:
        return []

    diffs = [abs((item_date(item) - target).days) for item in selected]
    best = min(diffs)
    return [item for item, diff in zip(selected, diffs) if diff == best]


def choose_s2_items(row, date_str):
    if pd.isna(date_str):
        return []

    geometry = aoi_features[row.aoi_id]["geometry"]
    items = search_items(S2_STAC, "sentinel-2-c1-l2a", geometry, date_str, days=2)
    if not items:
        items = search_items(S2_STAC, "sentinel-2-l2a", geometry, date_str, days=2)

    usable = []
    for item in items:
        needed = [asset_href(item, S2_ALIASES[k]) for k in ["green", "red", "nir", "swir"]]
        if all(needed):
            usable.append(item)

    if not usable:
        return []

    target = pd.Timestamp(date_str).normalize()
    diffs = np.array([abs((item_date(item) - target).days) for item in usable])
    best = int(diffs.min())
    chosen = [item for item, diff in zip(usable, diffs) if diff == best]
    chosen.sort(key=lambda x: float(x.properties.get("eo:cloud_cover", 100.0) or 100.0))
    return chosen[:8]


def load_s1(row, date_str, grid):
    items = choose_s1_items(row, date_str)
    vv_list, vh_list = [], []

    for item in items:
        vv = read_remote(asset_href(item, ["vv", "VV"]), grid)
        vh = read_remote(asset_href(item, ["vh", "VH"]), grid)
        if vv is not None and vh is not None:
            vv_list.append(vv)
            vh_list.append(vh)

    if not vv_list:
        return None

    vv = np.nanmedian(np.stack(vv_list), axis=0).astype(np.float32)
    vh = np.nanmedian(np.stack(vh_list), axis=0).astype(np.float32)
    return np.stack([vv, vh])


def to_reflectance(arr):
    arr = arr.astype(np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size and np.nanpercentile(finite, 95) > 2.0:
        arr = arr / 10000.0
    return np.clip(arr, 0.0, 1.5)


def load_s2(row, date_str, grid):
    items = choose_s2_items(row, date_str)
    stacks = {key: [] for key in ["green", "red", "nir", "swir"]}

    for item in items:
        arrays = {}
        ok = True
        for key in ["green", "red", "nir", "swir"]:
            arrays[key] = read_remote(asset_href(item, S2_ALIASES[key]), grid)
            if arrays[key] is None:
                ok = False
                break
        if not ok:
            continue

        scl = read_remote(asset_href(item, S2_ALIASES["scl"]), grid, Resampling.nearest)
        if scl is not None:
            bad = np.isin(np.rint(scl).astype(np.int16), [0, 1, 3, 8, 9, 10, 11])
        else:
            bad = np.zeros_like(arrays["green"], dtype=bool)

        for key in stacks:
            value = to_reflectance(arrays[key])
            value[bad] = np.nan
            stacks[key].append(value)

    if not stacks["green"]:
        return None

    result = []
    for key in ["green", "red", "nir", "swir"]:
        result.append(np.nanmedian(np.stack(stacks[key]), axis=0).astype(np.float32))
    return np.stack(result)


**Зачем:** эта ячейка заменяет весь прежний Earth Engine-код. Сцены выбираются по AOI, дате, орбите и поляризации; данные читаются как Cloud-Optimized GeoTIFF только в пределах нужной территории, поэтому не скачиваются многогигабайтные сцены целиком.


In [8]:
# 8. AUX, гидрография и reference

AUX_NAMES = ["slope", "hand", "occurrence", "seasonality", "max_extent", "builtup"]


def reproject_local_band(path, band, grid, resampling):
    dst = np.full((grid["height"], grid["width"]), np.nan, dtype=np.float32)
    with rasterio.open(path) as src:
        reproject(
            source=rasterio.band(src, band),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=grid["transform"],
            dst_crs=grid["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def load_aux(row, grid):
    path = ROOT / row.rasters_dir / "AUX_terrain_gsw.tif"
    bands = []
    for i in range(1, 7):
        method = Resampling.nearest if i in [5, 6] else Resampling.bilinear
        bands.append(reproject_local_band(path, i, grid, method))
    return np.stack(bands)


@lru_cache(maxsize=None)
def river_mask(aoi_id, res):
    grid = grid_for_aoi(aoi_id, res)
    shapes = []
    for feature in hydro_by_aoi.get(aoi_id, []):
        try:
            geom = shp_transform(TO_UTM, shape(feature["geometry"]))
            shapes.append((geom, 1))
        except Exception:
            pass
    if not shapes:
        return np.zeros((grid["height"], grid["width"]), dtype=np.float32)
    return rasterize(
        shapes,
        out_shape=(grid["height"], grid["width"]),
        transform=grid["transform"],
        fill=0,
        all_touched=True,
        dtype="uint8",
    ).astype(np.float32)


@lru_cache(maxsize=None)
def load_reference_by_id(pair_id):
    row = row_by_pid.loc[pair_id]
    if not row.has_reference:
        return None
    grid = work_grid(row)
    path = ROOT / row.reference_mask
    labels = []
    with rasterio.open(path) as src:
        for band in [1, 2, 3]:
            dst = np.zeros((grid["height"], grid["width"]), dtype=np.float32)
            reproject(
                source=rasterio.band(src, band),
                destination=dst,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=grid["transform"],
                dst_crs=grid["crs"],
                resampling=Resampling.average,
            )
            labels.append((dst >= 0.5).astype(np.uint8))
    return np.stack(labels)


def load_reference(row):
    return load_reference_by_id(row.pair_id)


**Зачем:** AUX даёт физику поймы — HAND, уклон, постоянную воду и застройку; OSM-гидрография добавляет близость к руслу. Reference здесь читается только как target для обучения и OOF-проверки, что соответствует назначению обучающей разметки.


In [9]:
# 9. Признаки и физическая модель

FEATURE_NAMES = []


def sigmoid_np(x):
    x = np.clip(x, -20, 20)
    return 1.0 / (1.0 + np.exp(-x))


def robust_z(arr, mask):
    arr = arr.astype(np.float32)
    valid = mask & np.isfinite(arr)
    if valid.sum() < 100:
        return np.zeros_like(arr, dtype=np.float32)
    p10, p50, p90 = np.nanpercentile(arr[valid], [10, 50, 90])
    scale = max(float(p90 - p10), 1e-4)
    out = (arr - p50) / scale
    return np.clip(np.nan_to_num(out), -4, 4).astype(np.float32)


def amplitude_to_db(arr):
    arr = np.asarray(arr, dtype=np.float32)
    positive = arr[np.isfinite(arr) & (arr > 0)]
    if positive.size == 0:
        return np.full_like(arr, np.nan)
    if np.nanpercentile(positive, 95) < 5:
        return arr
    return 20.0 * np.log10(np.maximum(arr, 1e-6))


def spectral_indices(s2):
    if s2 is None:
        return None
    green, red, nir, swir = s2
    eps = 1e-5
    ndwi = (green - nir) / (green + nir + eps)
    mndwi = (green - swir) / (green + swir + eps)
    ndvi = (nir - red) / (nir + red + eps)
    awei = 4.0 * (green - swir) - (0.25 * nir + 2.75 * swir)
    return np.stack([
        np.clip(ndwi, -1, 1),
        np.clip(mndwi, -1, 1),
        np.clip(ndvi, -1, 1),
        np.clip(awei, -2, 2) / 2.0,
    ]).astype(np.float32)


def pool_feature(arr, kernel):
    tensor = torch.from_numpy(np.nan_to_num(arr).astype(np.float32))[None, None]
    with torch.no_grad():
        out = F.avg_pool2d(tensor, kernel, stride=1, padding=kernel // 2)
    return out[0, 0].numpy()


def dilate_feature(arr, kernel):
    tensor = torch.from_numpy(arr.astype(np.float32))[None, None]
    with torch.no_grad():
        out = F.max_pool2d(tensor, kernel, stride=1, padding=kernel // 2)
    return out[0, 0].numpy()


def build_features(row):
    grid = work_grid(row)
    mask = aoi_mask(row.aoi_id, cfg.work_res_m)
    aux = load_aux(row, grid)

    slope = np.clip(np.nan_to_num(aux[0]), 0, 40) / 40.0
    hand = np.clip(np.nan_to_num(aux[1]), 0, 80) / 80.0
    occurrence = np.clip(np.nan_to_num(aux[2]), 0, 100) / 100.0
    seasonality = np.clip(np.nan_to_num(aux[3]), 0, 12) / 12.0
    max_extent = np.clip(np.nan_to_num(aux[4]), 0, 1)
    builtup = np.clip(np.nan_to_num(aux[5]), 0, 1)

    s1_pre = load_s1(row, row.date_pre_sar, grid)
    s1_peak = load_s1(row, row.date_peak_sar, grid)
    s1_ok = s1_pre is not None and s1_peak is not None

    if s1_pre is None:
        s1_pre = np.full((2, grid["height"], grid["width"]), np.nan, dtype=np.float32)
    if s1_peak is None:
        s1_peak = np.full((2, grid["height"], grid["width"]), np.nan, dtype=np.float32)

    pre_vv = robust_z(amplitude_to_db(s1_pre[0]), mask)
    pre_vh = robust_z(amplitude_to_db(s1_pre[1]), mask)
    peak_vv = robust_z(amplitude_to_db(s1_peak[0]), mask)
    peak_vh = robust_z(amplitude_to_db(s1_peak[1]), mask)

    dvv = np.clip(peak_vv - pre_vv, -3, 3) / 3.0
    dvh = np.clip(peak_vh - pre_vh, -3, 3) / 3.0
    pre_ratio = np.clip(pre_vv - pre_vh, -3, 3) / 3.0
    peak_ratio = np.clip(peak_vv - peak_vh, -3, 3) / 3.0

    s2_pre = load_s2(row, row.date_pre_opt, grid) if pd.notna(row.date_pre_opt) else None
    s2_peak = load_s2(row, row.date_peak_opt, grid) if pd.notna(row.date_peak_opt) else None
    idx_pre = spectral_indices(s2_pre)
    idx_peak = spectral_indices(s2_peak)

    opt_pre_ok = idx_pre is not None
    opt_peak_ok = idx_peak is not None

    if idx_pre is None:
        idx_pre = np.zeros((4, grid["height"], grid["width"]), dtype=np.float32)
    if idx_peak is None:
        idx_peak = np.zeros((4, grid["height"], grid["width"]), dtype=np.float32)

    idx_delta = np.clip(idx_peak - idx_pre, -2, 2) / 2.0

    sar_pre = sigmoid_np(-1.6 * (0.62 * pre_vv + 0.38 * pre_vh) - 0.05)
    sar_peak = sigmoid_np(-1.6 * (0.62 * peak_vv + 0.38 * peak_vh) - 0.05)
    sar_change = sigmoid_np(-2.0 * (0.6 * dvv + 0.4 * dvh))

    opt_pre = sigmoid_np(4.2 * (0.45 * idx_pre[0] + 0.55 * idx_pre[1] - 0.02))
    opt_peak = sigmoid_np(4.2 * (0.45 * idx_peak[0] + 0.55 * idx_peak[1] - 0.02))

    hydro = sigmoid_np((0.34 - hand) * 10.0) * np.exp(-2.2 * slope)
    river = river_mask(row.aoi_id, cfg.work_res_m)
    river_3 = dilate_feature(river, 3)
    river_9 = dilate_feature(river, 9)
    river_21 = dilate_feature(river, 21)
    hydro = np.clip(0.72 * hydro + 0.28 * river_21, 0, 1)

    if opt_pre_ok:
        water_pre_phys = 0.48 * sar_pre + 0.42 * opt_pre + 0.10 * occurrence
    else:
        water_pre_phys = 0.72 * sar_pre + 0.28 * occurrence

    if opt_peak_ok:
        water_peak_phys = 0.48 * sar_peak + 0.42 * opt_peak + 0.10 * occurrence
    else:
        water_peak_phys = 0.72 * sar_peak + 0.28 * occurrence

    water_pre_phys = np.clip(0.82 * water_pre_phys + 0.18 * max_extent, 0, 1)
    water_peak_phys = np.clip(0.82 * water_peak_phys + 0.18 * max_extent, 0, 1)

    permanent = occurrence >= 0.80
    flood_phys = water_peak_phys * (1.0 - water_pre_phys) * (0.55 + 0.45 * sar_change)
    flood_phys *= 0.55 + 0.45 * hydro
    flood_phys *= 1.0 - 0.92 * permanent.astype(np.float32)
    flood_phys *= 1.0 - 0.30 * builtup
    flood_phys = np.clip(flood_phys, 0, 1)

    peak_local_5 = pool_feature(water_peak_phys, 5)
    peak_local_15 = pool_feature(water_peak_phys, 15)
    flood_local_5 = pool_feature(flood_phys, 5)
    flood_local_15 = pool_feature(flood_phys, 15)

    channels = [
        pre_vv, pre_vh, peak_vv, peak_vh, dvv, dvh, pre_ratio, peak_ratio,
        *list(idx_pre), *list(idx_peak), *list(idx_delta),
        slope, hand, occurrence, seasonality, max_extent, builtup,
        river_3, river_9, river_21,
        sar_pre, sar_peak, sar_change,
        water_pre_phys, water_peak_phys, flood_phys,
        peak_local_5, peak_local_15, flood_local_5, flood_local_15,
        np.full_like(slope, float(s1_ok)),
        np.full_like(slope, float(opt_pre_ok)),
        np.full_like(slope, float(opt_peak_ok)),
    ]

    names = [
        "pre_vv", "pre_vh", "peak_vv", "peak_vh", "dvv", "dvh", "pre_ratio", "peak_ratio",
        "pre_ndwi", "pre_mndwi", "pre_ndvi", "pre_awei",
        "peak_ndwi", "peak_mndwi", "peak_ndvi", "peak_awei",
        "delta_ndwi", "delta_mndwi", "delta_ndvi", "delta_awei",
        "slope", "hand", "occurrence", "seasonality", "max_extent", "builtup",
        "river_3", "river_9", "river_21", "sar_pre", "sar_peak", "sar_change",
        "water_pre_phys", "water_peak_phys", "flood_phys",
        "peak_local_5", "peak_local_15", "flood_local_5", "flood_local_15",
        "s1_ok", "opt_pre_ok", "opt_peak_ok",
    ]

    x = np.stack(channels).astype(np.float32)
    x[:, ~mask] = 0
    x = np.nan_to_num(x, nan=0.0, posinf=4.0, neginf=-4.0)

    physics = np.stack([flood_phys, water_pre_phys, water_peak_phys]).astype(np.float32)
    physics[:, ~mask] = 0

    return x, physics, mask.astype(np.uint8), permanent.astype(np.uint8), names, {
        "s1": bool(s1_ok),
        "opt_pre": bool(opt_pre_ok),
        "opt_peak": bool(opt_peak_ok),
    }


**Зачем:** модель получает не только сырые спектральные признаки, но и физически осмысленные комбинации: падение SAR-сигнала, NDWI/MNDWI, HAND, уклон, постоянную воду и близость к руслу. Это особенно полезно при маленьком числе независимых паводков и делает решение менее зависимым от запоминания конкретной сцены.


In [10]:
# 10. Кэш признаков

status_rows = []

for _, row in tqdm(list(pairs.iterrows()), total=len(pairs), desc="Признаки"):
    path = CACHE / f"{row.pair_id}.npz"
    if not path.exists():
        x, physics, mask, permanent, names, status = build_features(row)
        np.savez(
            path,
            x=x.astype(np.float16),
            physics=physics.astype(np.float16),
            mask=mask,
            permanent=permanent,
            names=np.array(names),
            s1=np.uint8(status["s1"]),
            opt_pre=np.uint8(status["opt_pre"]),
            opt_peak=np.uint8(status["opt_peak"]),
        )
        del x, physics
        gc.collect()

    with np.load(path, allow_pickle=False) as data:
        status_rows.append({
            "pair_id": row.pair_id,
            "S1": bool(data["s1"]),
            "S2 pre": bool(data["opt_pre"]),
            "S2 peak": bool(data["opt_peak"]),
        })

status_df = pd.DataFrame(status_rows)
display(status_df)
print("Кэш готов:", CACHE)


Признаки:   0%|          | 0/11 [00:00<?, ?it/s]

,pair_id,S1,S2 pre,S2 peak
0,baseline_2018_09_low__blagoveshchensk,True,False,False
1,baseline_2018_09_low__konstantinovka,True,False,False
2,baseline_2018_09_low__svobodny,True,False,False
3,flood_2019_07_amur__belogorsk,True,True,True
4,flood_2019_07_amur__blagoveshchensk,True,False,False
5,flood_2019_07_amur__konstantinovka,True,False,False
6,flood_2019_07_amur__svobodny,True,True,True
7,flood_2021_06_amur__blagoveshchensk,True,True,True
8,flood_2021_06_amur__konstantinovka,True,True,True
9,flood_2021_06_amur__poyarkovo,True,True,True


Кэш готов: /content/hydrowatch_final/features_v7


**Зачем:** Sentinel читается из сети только один раз. Все последующие OOF-фолды работают с локальным кэшем, поэтому обучение можно перезапускать быстро и не тратить время повторно на спутниковые данные.


In [11]:
# 11. Выбор обучающих пикселей


def load_cache(pair_id):
    with np.load(CACHE / f"{pair_id}.npz", allow_pickle=False) as data:
        return {
            "x": data["x"].astype(np.float32),
            "physics": data["physics"].astype(np.float32),
            "mask": data["mask"].astype(bool),
            "permanent": data["permanent"].astype(bool),
        }


def choose_pixels(labels, physics, valid, n, rng):
    flat_valid = np.flatnonzero(valid.ravel())
    if flat_valid.size <= n:
        return flat_valid

    y_any = labels.any(axis=0).ravel()
    pos = flat_valid[y_any[flat_valid]]
    hard = flat_valid[(~y_any[flat_valid]) & (physics[0].ravel()[flat_valid] > 0.10)]

    n_pos = min(len(pos), int(n * 0.45))
    n_hard = min(len(hard), int(n * 0.30))
    n_random = n - n_pos - n_hard

    parts = []
    if n_pos:
        parts.append(rng.choice(pos, n_pos, replace=False))
    if n_hard:
        parts.append(rng.choice(hard, n_hard, replace=False))

    n_random = min(n_random, len(flat_valid))
    if n_random:
        parts.append(rng.choice(flat_valid, n_random, replace=False))

    out = np.concatenate(parts)
    rng.shuffle(out)
    return out


def build_train_matrix(train_rows, pixels_per_scene=None):
    if pixels_per_scene is None:
        pixels_per_scene = cfg.pixels_per_scene

    rng = np.random.default_rng(cfg.seed)
    xs, ys = [], []

    for _, row in train_rows.iterrows():
        cached = load_cache(row.pair_id)
        labels = load_reference(row)
        idx = choose_pixels(labels, cached["physics"], cached["mask"], pixels_per_scene, rng)

        flat_x = cached["x"].reshape(cached["x"].shape[0], -1).T
        flat_y = labels.reshape(3, -1).T
        xs.append(flat_x[idx])
        ys.append(flat_y[idx])

    x = np.concatenate(xs).astype(np.float32)
    y = np.concatenate(ys).astype(np.float32)

    order = rng.permutation(len(x))
    return x[order], y[order]


**Зачем:** вместо миллионов почти одинаковых сухих пикселей берём информативную выборку: реальные водные пиксели, трудные отрицательные примеры возле возможного затопления и случайный фон. Это одновременно ускоряет обучение и снижает перекос класса `flood`.


In [12]:
# 12. Модель

class PixelNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 160),
            nn.LayerNorm(160),
            nn.GELU(),
            nn.Dropout(0.12),
            nn.Linear(160, 160),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(160, 96),
            nn.GELU(),
            nn.Linear(96, 3),
        )

    def forward(self, x):
        return self.net(x)


def fit_model(train_rows, tag):
    x, y = build_train_matrix(train_rows)
    mean = x.mean(axis=0).astype(np.float32)
    std = x.std(axis=0).astype(np.float32)
    std = np.where(std < 1e-4, 1.0, std).astype(np.float32)
    x = (x - mean) / std

    pos = y.sum(axis=0)
    neg = len(y) - pos
    pos_weight = np.clip(neg / np.maximum(pos, 1.0), 1.0, 18.0).astype(np.float32)

    ds = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())

    model = PixelNet(x.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))

    model.train()
    for epoch in range(cfg.epochs):
        losses = []
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            probs = torch.sigmoid(logits)

            loss = criterion(logits, yb)
            loss = loss + 0.06 * F.relu(probs[:, 0] - probs[:, 2]).mean()
            loss = loss + 0.04 * (probs[:, 0] * probs[:, 1]).mean()
            loss.backward()
            opt.step()
            losses.append(float(loss.detach().cpu()))

        if epoch in [0, cfg.epochs - 1] or (epoch + 1) % 4 == 0:
            print(f"{tag}: эпоха {epoch + 1:02d}/{cfg.epochs}, loss={np.mean(losses):.4f}")

    state = {
        "model": model.state_dict(),
        "mean": mean,
        "std": std,
        "n_features": int(x.shape[1]),
    }
    torch.save(state, MODELS / f"{tag}.pt")

    del x, y, ds, loader
    gc.collect()
    return model, mean, std


@torch.no_grad()
def predict_scene(model, mean, std, pair_id):
    cached = load_cache(pair_id)
    x = cached["x"]
    valid = cached["mask"]
    c, h, w = x.shape
    flat = x.reshape(c, -1).T
    ids = np.flatnonzero(valid.ravel())
    out = np.zeros((3, h * w), dtype=np.float32)

    model.eval()
    for start in range(0, len(ids), cfg.predict_batch):
        part = ids[start:start + cfg.predict_batch]
        xb = (flat[part] - mean) / std
        xb = torch.from_numpy(xb.astype(np.float32)).to(DEVICE)
        out[:, part] = torch.sigmoid(model(xb)).cpu().numpy().T

    return out.reshape(3, h, w), cached


**Зачем:** модель маленькая и быстро обучается на GPU, но получает уже богатые физические и многомасштабные признаки. В loss дополнительно заложены два логичных ограничения: flood не должен быть вероятнее peak-water и не должен одновременно считаться водой `до` события.


In [13]:
# 13. Leave-one-event-out OOF

ref_pairs = pairs[pairs["has_reference"]].copy()
events = list(ref_pairs["event_id"].drop_duplicates())

for hold_event in events:
    hold_rows = ref_pairs[ref_pairs["event_id"] == hold_event]
    train_rows = ref_pairs[ref_pairs["event_id"] != hold_event]

    print("\nHoldout:", hold_event, "| train:", len(train_rows), "| valid:", len(hold_rows))
    model, mean, std = fit_model(train_rows, f"fold_{hold_event}")

    for _, row in hold_rows.iterrows():
        nn_probs, cached = predict_scene(model, mean, std, row.pair_id)
        np.savez(
            OOF / f"{row.pair_id}.npz",
            nn=nn_probs.astype(np.float16),
            physics=cached["physics"].astype(np.float16),
        )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print("OOF готов для", len(list(OOF.glob("*.npz"))), "пар")



Holdout: baseline_2018_09_low | train: 8 | valid: 3
fold_baseline_2018_09_low: эпоха 01/12, loss=0.6314
fold_baseline_2018_09_low: эпоха 04/12, loss=0.3564
fold_baseline_2018_09_low: эпоха 08/12, loss=0.3311
fold_baseline_2018_09_low: эпоха 12/12, loss=0.3157

Holdout: flood_2019_07_amur | train: 7 | valid: 4
fold_flood_2019_07_amur: эпоха 01/12, loss=0.6187
fold_flood_2019_07_amur: эпоха 04/12, loss=0.3243
fold_flood_2019_07_amur: эпоха 08/12, loss=0.2997
fold_flood_2019_07_amur: эпоха 12/12, loss=0.2846

Holdout: flood_2021_06_amur | train: 8 | valid: 3
fold_flood_2021_06_amur: эпоха 01/12, loss=0.6051
fold_flood_2021_06_amur: эпоха 04/12, loss=0.3238
fold_flood_2021_06_amur: эпоха 08/12, loss=0.3028
fold_flood_2021_06_amur: эпоха 12/12, loss=0.2893

Holdout: flood_2021_08_zeya | train: 10 | valid: 1
fold_flood_2021_08_zeya: эпоха 01/12, loss=0.5925
fold_flood_2021_08_zeya: эпоха 04/12, loss=0.3517
fold_flood_2021_08_zeya: эпоха 08/12, loss=0.3276
fold_flood_2021_08_zeya: эпоха 12/1

**Зачем:** это главный анти-overfit слой. Например, прогноз для паводка 2019 строится только по другим событиям; модель не может просто запомнить маску этого паводка или соседний AOI из того же события.


In [14]:
# 14. Калибровка по OOF


@lru_cache(maxsize=None)
def load_oof(pair_id):
    with np.load(OOF / f"{pair_id}.npz", allow_pickle=False) as data:
        return data["nn"].astype(np.float32), data["physics"].astype(np.float32)


def dice_score(pred, true):
    pred = pred.astype(bool)
    true = true.astype(bool)
    inter = np.logical_and(pred, true).sum()
    denom = pred.sum() + true.sum()
    if denom == 0:
        return 1.0
    return 2.0 * inter / denom


def area_quality(pred, true):
    p = float(pred.sum())
    t = float(true.sum())
    floor = 500.0
    return math.exp(-abs(p - t) / max(t, floor))


def candidate_score(target, weight, threshold):
    values = []
    for _, row in ref_pairs.iterrows():
        nn_probs, physics = load_oof(row.pair_id)
        labels = load_reference(row)
        prob = weight * nn_probs[target] + (1.0 - weight) * physics[target]
        prob = prob[::4, ::4]
        pred = prob >= threshold
        if target == 0 and row.event_kind == "baseline":
            pred[:] = False
        valid = aoi_mask(row.aoi_id, cfg.work_res_m)[::4, ::4]
        pred &= valid
        true = labels[target][::4, ::4].astype(bool) & valid
        values.append(0.72 * dice_score(pred, true) + 0.28 * area_quality(pred, true))
    return float(np.mean(values))


def tune_target(target):
    best = None
    for weight in [0.45, 0.60, 0.75, 0.90]:
        for threshold in np.arange(0.20, 0.81, 0.04):
            score = candidate_score(target, weight, float(threshold))
            if best is None or score > best[0]:
                best = (score, weight, float(threshold))
    return {"score": best[0], "weight": best[1], "threshold": best[2]}


calibration = {
    "flood": tune_target(0),
    "water_pre": tune_target(1),
    "water_peak": tune_target(2),
}

print(json.dumps(calibration, ensure_ascii=False, indent=2))


{
  "flood": {
    "score": 0.2287406133021955,
    "weight": 0.9,
    "threshold": 0.6800000000000002
  },
  "water_pre": {
    "score": 0.6742189951499101,
    "weight": 0.45,
    "threshold": 0.76
  },
  "water_peak": {
    "score": 0.5325972293018932,
    "weight": 0.45,
    "threshold": 0.7200000000000002
  }
}


**Зачем:** у нас нет видимого leaderboard, поэтому вес нейросети/физики и пороги выбираются только по честным OOF-прогнозам. Один общий набор параметров используется для всех сцен — никакой ручной подгонки конкретных `pair_id`.


In [15]:
# 15. Финальная модель для новых пар

sample_rows = pairs[pairs["pair_id"].isin(sample["pair_id"])].copy()
new_rows = sample_rows[~sample_rows["has_reference"]]
final_model = None
final_mean = None
final_std = None

if len(new_rows):
    print("Новых пар без reference:", len(new_rows))
    final_model, final_mean, final_std = fit_model(ref_pairs, "final_all_events")

    for _, row in new_rows.iterrows():
        nn_probs, cached = predict_scene(final_model, final_mean, final_std, row.pair_id)
        np.savez(
            OOF / f"{row.pair_id}.npz",
            nn=nn_probs.astype(np.float16),
            physics=cached["physics"].astype(np.float16),
        )
else:
    print("В sample_submission все пары имеют OOF-прогноз — дополнительное обучение не требуется")


В sample_submission все пары имеют OOF-прогноз — дополнительное обучение не требуется


**Зачем:** открытые пары получают строго OOF-прогноз. Если организаторы дадут новые/private пары без разметки, ноутбук автоматически обучит одну финальную модель на всех доступных событиях и применит те же OOF-калиброванные параметры.


In [27]:
# 16. Финальные маски

@lru_cache(maxsize=None)
def blended_probs_by_id(pair_id):
    nn_probs, physics = load_oof(pair_id)
    probs = []

    for name, idx in [("flood", 0), ("water_pre", 1), ("water_peak", 2)]:
        w = calibration[name]["weight"]
        probs.append((w * nn_probs[idx] + (1 - w) * physics[idx]).astype(np.float32))

    return tuple(probs)


@lru_cache(maxsize=None)
def valid_by_id(pair_id):
    return load_cache(pair_id)["mask"].astype(bool)


def smooth_prob(prob, kernel=3):
    x = torch.from_numpy(prob.astype(np.float32))[None, None]

    with torch.no_grad():
        y = F.avg_pool2d(x, kernel, stride=1, padding=kernel // 2)

    return y[0, 0].numpy()


def estimate_pixels(row, target, prob, valid):
    if target == 0 and row.event_kind == "baseline":
        return 0

    calib_rows = ref_pairs[ref_pairs["event_id"] != row.event_id].copy()

    if target == 0:
        calib_rows = calib_rows[calib_rows["event_kind"] != "baseline"]
        thresholds = [0.28, 0.38, 0.48, 0.58, 0.68]
        max_fraction = 0.03
    else:
        thresholds = [0.35, 0.45, 0.55, 0.65, 0.75]
        max_fraction = 0.12

    estimates = []
    true_fractions = []

    for threshold in thresholds:
        ratios = []

        for _, ref_row in calib_rows.iterrows():
            ref_prob = blended_probs_by_id(ref_row.pair_id)[target]
            ref_valid = valid_by_id(ref_row.pair_id)

            true = load_reference(ref_row)[target].astype(bool) & ref_valid

            pred_count = int(((ref_prob >= threshold) & ref_valid).sum())
            true_count = int(true.sum())

            if true_count > 0:
                true_fractions.append(true_count / max(int(ref_valid.sum()), 1))

            if pred_count >= 25 and true_count > 0:
                ratios.append(true_count / pred_count)

        current_count = int(((prob >= threshold) & valid).sum())

        if ratios and current_count > 0:
            scale = float(
                np.exp(
                    np.median(
                        np.log(np.clip(ratios, 0.03, 30.0))
                    )
                )
            )
            estimates.append(current_count * scale)

    soft_ratios = []

    for _, ref_row in calib_rows.iterrows():
        ref_prob = blended_probs_by_id(ref_row.pair_id)[target]
        ref_valid = valid_by_id(ref_row.pair_id)

        true = load_reference(ref_row)[target].astype(bool) & ref_valid

        true_count = int(true.sum())
        soft_count = float(((ref_prob ** 2) * ref_valid).sum())

        if soft_count > 25 and true_count > 0:
            soft_ratios.append(true_count / soft_count)

    current_soft = float(((prob ** 2) * valid).sum())

    if soft_ratios and current_soft > 0:
        scale = float(
            np.exp(
                np.median(
                    np.log(np.clip(soft_ratios, 0.03, 30.0))
                )
            )
        )
        estimates.append(current_soft * scale)

    if estimates:
        target_pixels = int(round(float(np.median(estimates))))
    else:
        name = ["flood", "water_pre", "water_peak"][target]
        threshold = calibration[name]["threshold"]
        target_pixels = int(((prob >= threshold) & valid).sum())

    if target == 0 and true_fractions:
        low_fraction = float(np.quantile(true_fractions, 0.10))
        target_pixels = max(
            target_pixels,
            int(valid.sum() * low_fraction)
        )

    target_pixels = max(0, target_pixels)
    target_pixels = min(target_pixels, int(valid.sum() * max_fraction))

    return target_pixels


def topk_mask(score, valid, k):
    mask = np.zeros_like(valid, dtype=bool)
    idx = np.flatnonzero(valid.ravel())

    if k <= 0 or idx.size == 0:
        return mask

    k = min(int(k), idx.size)
    values = score.ravel()[idx]

    if k == idx.size:
        chosen = idx
    else:
        chosen = idx[np.argpartition(values, -k)[-k:]]

    mask.ravel()[chosen] = True
    return mask


def final_masks(row):
    cached = load_cache(row.pair_id)

    physics = cached["physics"]
    valid = cached["mask"]
    permanent = cached["permanent"] & valid
    x = cached["x"]

    p_flood, p_pre, p_peak = blended_probs_by_id(row.pair_id)

    p_flood = 0.85 * p_flood + 0.15 * smooth_prob(p_flood)
    p_pre = 0.88 * p_pre + 0.12 * smooth_prob(p_pre)
    p_peak = 0.88 * p_peak + 0.12 * smooth_prob(p_peak)

    hand = np.clip(x[21], 0, 1)
    slope = np.clip(x[20], 0, 1)
    river = np.clip(x[28], 0, 1)
    sar_change = np.clip(x[31], 0, 1)

    hydro = np.exp(-2.8 * hand)
    hydro *= 1 - 0.45 * slope
    hydro = np.clip(hydro + 0.22 * river, 0, 1)

    pre_score = np.clip(
        0.90 * p_pre + 0.07 * physics[1] + 0.03 * x[22],
        0,
        1
    )

    peak_score = np.clip(
        0.90 * p_peak + 0.07 * physics[2] + 0.03 * x[22],
        0,
        1
    )

    flood_score = p_flood.copy()
    flood_score *= 0.70 + 0.30 * p_peak
    flood_score *= 0.76 + 0.24 * (1 - p_pre)
    flood_score *= 0.72 + 0.28 * physics[0]
    flood_score *= 0.76 + 0.24 * sar_change
    flood_score *= 0.76 + 0.24 * hydro
    flood_score = 0.88 * flood_score + 0.12 * smooth_prob(flood_score)

    n_pre = estimate_pixels(row, 1, p_pre, valid)
    n_peak = estimate_pixels(row, 2, p_peak, valid)

    pre = topk_mask(pre_score, valid, n_pre)
    peak = topk_mask(peak_score, valid, n_peak)

    if row.event_kind == "baseline":
        flood = np.zeros_like(valid, dtype=bool)
    else:
        flood_valid = valid & ~permanent
        n_flood = estimate_pixels(row, 0, p_flood, flood_valid)
        flood = topk_mask(flood_score, flood_valid, n_flood)

    peak |= flood

    pre &= valid
    peak &= valid
    flood &= valid
    flood &= ~permanent

    return (
        flood.astype(np.uint8),
        pre.astype(np.uint8),
        peak.astype(np.uint8)
    )


def to_output_grid(mask, row):
    src_grid = work_grid(row)
    dst_grid = out_grid(row)

    dst = np.zeros(
        (dst_grid["height"], dst_grid["width"]),
        dtype=np.uint8
    )

    reproject(
        source=mask,
        destination=dst,
        src_transform=src_grid["transform"],
        src_crs=src_grid["crs"],
        dst_transform=dst_grid["transform"],
        dst_crs=dst_grid["crs"],
        resampling=Resampling.nearest
    )

    dst[~aoi_mask(row.aoi_id, cfg.out_res_m)] = 0
    return dst


def write_flood_tif(path, mask, row):
    grid = out_grid(row)

    profile = {
        "driver": "GTiff",
        "height": grid["height"],
        "width": grid["width"],
        "count": 1,
        "dtype": "uint8",
        "crs": grid["crs"],
        "transform": grid["transform"],
        "compress": "deflate",
        "predictor": 2,
        "nodata": 0
    }

    with rasterio.open(path, "w", **profile) as dst:
        dst.write(mask.astype(np.uint8), 1)

**Зачем:** после ML применяются только общие физические ограничения: постоянная вода не считается новым flood, flood должен лежать внутри peak-water, а контрольный baseline не должен давать ложное наводнение. Никаких правил по конкретному городу или конкретной строке сабмита здесь нет.


In [28]:
# 17. Submission и GeoTIFF

rows = []

for pair_id in sample["pair_id"]:
    row = row_by_pid.loc[pair_id].copy()
    row["pair_id"] = pair_id

    flood20, pre20, peak20 = final_masks(row)

    flood10 = to_output_grid(flood20, row)
    pre10 = to_output_grid(pre20, row)
    peak10 = to_output_grid(peak20, row)

    peak10 = np.maximum(peak10, flood10).astype(np.uint8)
    flood10 = (flood10 & (peak10 > 0)).astype(np.uint8)

    write_flood_tif(PRED / f"{pair_id}_flood.tif", flood10, row)

    pixel_ha = (cfg.out_res_m * cfg.out_res_m) / 10000.0

    flood_ha = float(flood10.sum() * pixel_ha)
    pre_ha = float(pre10.sum() * pixel_ha)
    peak_ha = float(peak10.sum() * pixel_ha)

    rows.append({
        "pair_id": pair_id,
        "flood_ha": round(flood_ha, 2),
        "water_pre_ha": round(pre_ha, 2),
        "water_peak_ha": round(max(peak_ha, flood_ha), 2)
    })

submission = pd.DataFrame(rows)
submission = sample[["pair_id"]].merge(submission, on="pair_id", how="left")

submission.to_csv(OUT / "submission.csv", index=False)

(OUT / "calibration.json").write_text(
    json.dumps(calibration, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

(OUT / "config.json").write_text(
    json.dumps(asdict(cfg), ensure_ascii=False, indent=2),
    encoding="utf-8"
)

display(submission)

,pair_id,flood_ha,water_pre_ha,water_peak_ha
0,baseline_2018_09_low__blagoveshchensk,0.00,7075.30,8150.42
1,baseline_2018_09_low__konstantinovka,0.00,6965.99,6988.77
2,baseline_2018_09_low__svobodny,0.00,3815.92,4288.46
3,flood_2019_07_amur__belogorsk,117.40,208.52,212.98
4,flood_2019_07_amur__blagoveshchensk,710.28,10074.84,7282.14
5,flood_2019_07_amur__konstantinovka,598.80,9923.71,5954.79
6,flood_2019_07_amur__svobodny,624.70,858.56,5167.61
7,flood_2021_06_amur__blagoveshchensk,2945.94,3339.15,14452.34
8,flood_2021_06_amur__konstantinovka,1440.34,88.44,15789.84
9,flood_2021_06_amur__poyarkovo,868.56,1887.83,5580.16


**Зачем:** площади считаются только из собственных финальных масок. Поэтому `submission.csv` и передаваемые организаторам GeoTIFF всегда согласованы между собой и воспроизводятся одним запуском ноутбука.


In [29]:
# 18. Проверка результата

assert list(submission.columns) == ["pair_id", "flood_ha", "water_pre_ha", "water_peak_ha"]
assert submission["pair_id"].tolist() == sample["pair_id"].tolist()
assert not submission.isna().any().any()
assert (submission[["flood_ha", "water_pre_ha", "water_peak_ha"]] >= 0).all().all()
assert (submission["flood_ha"] <= submission["water_peak_ha"] + 1e-9).all()

for pair_id in sample["pair_id"]:
    row = row_by_pid.loc[pair_id]
    grid = out_grid(row)
    path = PRED / f"{pair_id}_flood.tif"

    with rasterio.open(path) as src:
        arr = src.read(1)
        assert src.dtypes[0] == "uint8"
        assert src.height == grid["height"] and src.width == grid["width"]
        assert src.crs == rasterio.crs.CRS.from_string(grid["crs"])
        assert src.transform.almost_equals(grid["transform"], precision=1e-6)
        assert set(np.unique(arr)).issubset({0, 1})

    csv_area = float(submission.loc[submission["pair_id"] == pair_id, "flood_ha"].iloc[0])
    tif_area = round(float(arr.sum()) * 0.01, 2)
    assert abs(csv_area - tif_area) < 0.011

print("Все проверки пройдены")
print("submission.csv:", OUT / "submission.csv")
print("GeoTIFF:", PRED)


Все проверки пройдены
submission.csv: /content/hydrowatch_final/result/submission.csv
GeoTIFF: /content/hydrowatch_final/result/predictions


**Зачем:** перед сдачей автоматически ловим самые неприятные технические ошибки: неправильные `pair_id`, NaN, отрицательные площади, неверный CRS/transform, не-binary TIFF и расхождение площади между CSV и маской.


In [30]:
# 19. Архив для сдачи

package = WORK / "HydroWatch_FINAL_SUBMISSION.zip"

if package.exists():
    package.unlink()

with zipfile.ZipFile(package, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(OUT / "submission.csv", "submission.csv")
    archive.write(OUT / "calibration.json", "calibration.json")
    archive.write(OUT / "config.json", "config.json")
    for path in sorted(PRED.glob("*_flood.tif")):
        archive.write(path, f"predictions/{path.name}")

print("Готово:", package)
print("Размер, МБ:", round(package.stat().st_size / 2**20, 2))

try:
    from google.colab import files
    files.download(str(package))
except Exception:
    pass


Готово: /content/hydrowatch_final/HydroWatch_FINAL_SUBMISSION.zip
Размер, МБ: 0.18


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Зачем:** последняя ячейка собирает ровно те артефакты, которые нужны для отправки и проверки воспроизводимости. Главный файл для платформы — `submission.csv`; папка `predictions` содержит flood-маски для организаторов.
